### Chapter 10: Tool Engineering
### Topic 2: Tool-Calling Under the Hood — The Actual Message Mechanics

# End-to-End Flow of Conversation & Message Flow

This chapter explains how an LLM remembers a conversation.

The most important idea is that **an LLM has no memory of previous API calls**. The application maintains the conversation history and sends it back to the model on every request.

The program performs the following steps in order:

1. User sends a message.
2. The application stores the message in a `messages` list.
3. The application sends the entire `messages` list to the LLM.
4. The LLM generates a response.
5. The application appends the response to the `messages` list.
6. The next user message is added.
7. The updated `messages` list is sent again.
8. The conversation continues in the same way.

---

### Why Do We Need a Messages List?

Suppose a customer asks:

> **"What is premature FD withdrawal?"**

The LLM replies:

> **"It means closing an FD before its maturity date."**

Now the customer asks:

> **"What is the penalty?"**

How does the LLM know that **"penalty"** refers to **premature FD withdrawal**?

Does the LLM remember the previous question?

No.

An LLM has **no memory** between API calls.

If nothing else were done, the model would only see:

```text
What is the penalty?
```

It would have no idea what the customer is talking about.

---

### Then Who Remembers the Conversation?

The **application** remembers everything.

Instead of relying on the LLM, the application stores every conversation inside a variable called:

```python
messages
```

Think of this as the **conversation history**.

---

### How Does a Conversation Start?

Initially, the customer asks:

> **"What is premature FD withdrawal?"**

The application creates the messages list.

```python
messages = [
    {
        "role": "user",
        "content": "What is premature FD withdrawal?"
    }
]
```

At this moment, the conversation contains only one message.

---

### What Is a Message?

Every conversation is made up of **messages**.

Each message contains two important fields.

```text
Role

Content
```

Example:

```text
Role

user

------------------------

Content

What is premature FD withdrawal?
```

The **role** tells the model who spoke.

The **content** contains the actual text.

---

### First API Call

The application sends the entire messages list.

```text
messages

------------------------

User

What is premature FD withdrawal?
```

This is everything the LLM knows.

The LLM generates a response.

```text
Premature withdrawal means closing an FD before maturity.
```

---

### What Happens After the LLM Replies?

The LLM does not remember its own answer.

The application saves it.

Now the messages list becomes:

```text
messages

------------------------

User

What is premature FD withdrawal?

------------------------

Assistant

Premature withdrawal means closing an FD before maturity.
```

Notice something important.

The application—not the LLM—is building the conversation history.

---

### What Happens When the User Asks Another Question?

The customer now asks:

> **"What is the penalty?"**

The application appends another message.

```text
messages

------------------------

User

What is premature FD withdrawal?

------------------------

Assistant

Premature withdrawal means closing an FD before maturity.

------------------------

User

What is the penalty?
```

The conversation has grown.

---

### What Does the Application Send Now?

Many beginners think the application sends only:

```text
What is the penalty?
```

That is incorrect.

The application sends the **entire messages list** again.

```text
User

What is premature FD withdrawal?

------------------------

Assistant

Premature withdrawal means closing an FD before maturity.

------------------------

User

What is the penalty?
```

Now the LLM understands exactly what "penalty" refers to.

---

### Why Is the Entire Conversation Sent Every Time?

The LLM has **no server-side memory**.

Every API request is independent.

The application must provide all necessary context.

Think of every API call as a completely new conversation.

The only reason the LLM appears to remember earlier messages is because the application keeps sending them again.

---

### How Does the Conversation Continue?

Suppose the LLM replies:

```text
Premature withdrawal incurs a 1% penalty.
```

The application stores this response.

Now the conversation becomes:

```text
User

What is premature FD withdrawal?

------------------------

Assistant

Premature withdrawal means closing an FD before maturity.

------------------------

User

What is the penalty?

------------------------

Assistant

Premature withdrawal incurs a 1% penalty.
```

Every new message is simply appended to the list.

---

### Why Does the Messages List Keep Growing?

Each conversation adds more messages.

Example:

```text
User

Assistant

User

Assistant

User

Assistant

...
```

As the conversation becomes longer, the messages list grows larger.

This growing history is what allows the LLM to answer follow-up questions correctly.

---

### Does the LLM Really Remember Anything?

No.

Imagine deleting the messages list before making the next API call.

The application sends only:

```text
What is the penalty?
```

The LLM will most likely ask:

> **"Penalty for what?"**

because it has lost all previous context.

This proves that the memory belongs to the application, not the LLM.

---

### Why Is This Important for Tool Calling?

Tool Calling is simply another part of the conversation.

The tool request and tool result are also stored inside the same messages list.

In the next topic, we'll see how Tool Calling adds new message types to this conversation history.

---

### Production Engineer's Perspective

Always remember these rules:

- The LLM has no memory between API calls.
- The application stores the conversation history.
- Every API call sends the complete messages list.
- Every new message is appended to the conversation.
- If a message is missing from the list, the LLM has no way of knowing it ever happened.

A useful mental model is:

```text
Application

Conversation Memory

↓

messages List

↓

LLM
```

The **messages list is the LLM's memory**.

Every production AI framework—whether OpenAI Agents, Claude, LangChain, LangGraph, CrewAI, or AutoGen—uses this same fundamental principle, even if it hides the implementation details.

---

### Where Does Tool Calling Fit?

In the previous topic we learned that the application stores every conversation inside the `messages` list.

Tool Calling does not introduce a new conversation.

It simply adds more messages to the existing conversation.

Everything still happens inside the same `messages` list.

---

### Initial Conversation

Suppose the customer asks:

> **"Check my FD reference FD123456."**

Initially the conversation contains only one message.

```text
messages

------------------------

User

Check my FD reference FD123456.
```

The application sends this conversation to the LLM.

---

### What Else Is Sent to the LLM?

Besides the conversation, the application also sends:

- System Prompt
- Tool Schemas

The LLM now knows:

- what the customer asked
- what tools are available
- when those tools should be used

---

### How Does the LLM Decide to Call a Tool?

The LLM reads:

- User message
- System prompt
- Tool descriptions

It reasons:

- The customer provided an FD reference.
- I cannot validate it myself.
- A tool named `validate_fd_reference` is available.
- I should request that tool.

At this point, **nothing has been executed**.

The LLM is only making a decision.

---

### What Does the LLM Return?

Instead of returning a normal answer, the LLM returns a **tool request**.

Example:

```text
Tool

validate_fd_reference

Arguments

reference_number = FD123456
```

This is called a **tool_use** block.

Think of it as saying:

> "Please execute this tool before I continue."

---

### Why Doesn't the LLM Return the Final Answer?

Because it doesn't know the answer yet.

It needs information from the application.

Without executing the tool, the LLM cannot verify the FD reference.

---

### What Is stop_reason?

Every model response contains a field called **stop_reason**.

It tells the application why the model stopped generating.

Example:

```text
stop_reason

tool_use
```

This means:

> "I am waiting for a tool result."

Another example:

```text
stop_reason

end_turn
```

This means:

> "I have finished answering."

The application always checks `stop_reason` before deciding what to do next.

---

### Why Is stop_reason Important?

Suppose the application assumes every response contains a tool call.

But the model actually returned:

```text
Premature withdrawal incurs a penalty.
```

There is no tool request.

If the application still tries to execute a tool, it will fail.

Checking `stop_reason` prevents this mistake.

---

### What Is tool_use?

A **tool_use** block contains everything needed to execute a tool.

Example:

```text
Tool Name

validate_fd_reference

Arguments

reference_number = FD123456

Tool ID

tool_001
```

Notice the **Tool ID**.

Every tool request receives a unique identifier.

---

### Why Do We Need tool_use_id?

Suppose the LLM requests two tools.

```text
Tool 1

validate_fd_reference

ID = tool_001

------------------------

Tool 2

search_knowledge_base

ID = tool_002
```

Later the application executes both tools.

How does the LLM know which result belongs to which request?

The answer is **tool_use_id**.

---

### What Happens After the Tool Request?

The application stores the LLM's response.

Now the conversation becomes:

```text
User

Check my FD reference FD123456.

------------------------

Assistant

Request

validate_fd_reference()

ID = tool_001
```

This step is extremely important.

The assistant's response must be stored before executing the tool.

---

### Why Must We Store the Assistant's Tool Request?

Suppose we skip this step.

The conversation becomes:

```text
User

Check my FD reference FD123456.

------------------------

Tool Result

Reference Valid
```

The LLM now sees a tool result.

But it has no record of ever requesting that tool.

The conversation becomes inconsistent.

This is one of the most common Tool Calling bugs.

---

### What Happens After the Tool Executes?

The application runs:

```python
validate_fd_reference("FD123456")
```

Suppose the function returns:

```text
Reference

FD123456

Status

Valid
```

The application packages this as a **tool_result**.

Example:

```text
Tool Result

tool_use_id = tool_001

Result

Reference Valid
```

Notice that the **tool_use_id** is included.

This connects the result to the original request.

---

### Why Is tool_use_id Important?

Suppose two tools execute simultaneously.

```text
Tool Request

tool_001

------------------------

Tool Request

tool_002
```

Later the application returns:

```text
Result

tool_use_id = tool_002

------------------------

Result

tool_use_id = tool_001
```

The order no longer matters.

The LLM uses the **tool_use_id** to correctly match each result with the corresponding request.

---

### What Does the Conversation Look Like Now?

After appending the tool result, the conversation becomes:

```text
User

Check my FD reference FD123456.

------------------------

Assistant

Request

validate_fd_reference()

------------------------

User

Tool Result

Reference Valid
```

Notice something surprising.

The **tool result is added as another message**.

Nothing special happens.

The conversation simply grows.

---

### What Happens Next?

The application sends the **entire updated conversation** back to the LLM.

Now the LLM sees:

- Customer request
- Its own tool request
- Tool result

Since it finally has the required information, it generates the final answer.

Example:

```text
Your FD reference FD123456 is valid.
```

---

### Complete Tool Calling Conversation

```text
User

Check my FD reference FD123456.

------------------------

Assistant

Tool Request

validate_fd_reference()

------------------------

User

Tool Result

Reference Valid

------------------------

Assistant

Your FD reference is valid.
```

This entire conversation is stored inside the same `messages` list.

---

### Production Engineer's Perspective

Always remember these rules:

- Tool Calling is just another conversation.
- Tool requests are stored as assistant messages.
- Tool results are stored as user messages.
- Every tool request has a unique `tool_use_id`.
- Every tool result references the corresponding `tool_use_id`.
- The application always checks `stop_reason` before executing tools.
- The assistant's tool request must always be appended before the tool result.
- The entire updated conversation is sent back to the LLM after every step.

The conversation never resets.

The `messages` list keeps growing, and Tool Calling simply adds more messages to that same conversation history.

---

# End-to-End Flow of Production Agent Loop

This chapter explains how a production AI agent repeatedly interacts with the LLM until it reaches the final answer.

The most important idea is that **an agent is simply a loop**. Every iteration sends the current conversation to the LLM, executes any requested tools, appends the results, and repeats until the model produces a final answer.

The program performs the following steps in order:

1. Initialize the conversation.
2. Start the agent loop.
3. Send the complete conversation to the LLM.
4. Check whether the LLM requested a tool.
5. If required, execute the tool.
6. Append the tool result.
7. Repeat the loop.
8. Stop when the LLM generates the final answer or when the maximum number of steps is reached.

---

### Why Do We Need an Agent Loop?

Suppose the customer asks:

> **"Check my FD reference and tell me the premature withdrawal penalty."**

Can the LLM answer immediately?

No.

It first needs to:

- Validate the FD reference.
- Search the knowledge base.
- Read both results.
- Generate the final answer.

Since multiple steps are involved, a single API call is not enough.

The application must repeatedly communicate with the LLM.

This repeated communication is called the **Agent Loop**.

---

### How Does the Agent Loop Start?

Initially the application creates the conversation.

```text
messages

------------------------

User

Check my FD reference FD123456 and tell me the premature withdrawal penalty.
```

The application now starts the loop.

Example:

```python
for step in range(max_steps):
```

Each iteration represents one interaction with the LLM.

---

### Step 1: Send the Conversation

The application sends:

- System Prompt
- Tool Schemas
- Complete Messages List

The LLM now has everything it needs to make a decision.

---

### Step 2: Check stop_reason

The LLM responds.

The application immediately checks:

```text
stop_reason
```

Two situations are possible.

#### Case 1

```text
stop_reason

tool_use
```

The LLM wants a tool.

The loop continues.

#### Case 2

```text
stop_reason

end_turn
```

The LLM has produced the final answer.

The loop ends.

This is why checking **stop_reason** is extremely important.

---

### Step 3: Append the Assistant Response

Suppose the LLM requests:

```text
validate_fd_reference()
```

The application immediately stores this response.

```text
User

Check FD123456

------------------------

Assistant

Request

validate_fd_reference()
```

Never skip this step.

The conversation must contain the assistant's tool request.

---

### Step 4: Execute the Tool

The dispatch layer identifies the requested tool.

Example:

```python
if tool_name == "validate_fd_reference":
    validate_fd_reference(...)
```

The application now executes the real Python function.

The LLM does nothing during this step.

---

### Step 5: Append the Tool Result

Suppose the tool returns:

```text
Reference Valid
```

The application stores this result.

```text
User

Tool Result

Reference Valid
```

The conversation has grown again.

---

### Step 6: Repeat the Loop

The updated conversation is sent back to the LLM.

Now the LLM can:

- Generate the final answer.
- Request another tool.

Suppose the model now requests:

```text
search_knowledge_base()
```

The application repeats exactly the same process.

The loop continues until the model has everything it needs.

---

### Complete Agent Loop Example

```text
Customer

Check my FD reference and tell me the penalty.

------------------------

LLM

Request

validate_fd_reference()

------------------------

Application

Execute Tool

------------------------

Application

Append Tool Result

------------------------

LLM

Request

search_knowledge_base()

------------------------

Application

Execute Tool

------------------------

Application

Append Tool Result

------------------------

LLM

Generate Final Answer
```

The same loop repeats until the work is complete.

---

### Why Do We Need max_steps?

Suppose something goes wrong.

The LLM repeatedly requests:

```text
Tool

search_knowledge_base()

------------------------

Tool

search_knowledge_base()

------------------------

Tool

search_knowledge_base()
```

The conversation never ends.

Without protection, the application would continue forever.

To prevent this, every agent has a limit.

Example:

```python
max_steps = 5
```

After five iterations, the application stops.

This prevents:

- Infinite loops.
- Extremely high API costs.
- Very long response times.

---

### Why Does Latency Increase?

Each iteration requires another API call.

Without tools:

```text
User

LLM

Answer
```

One model call.

With one tool:

```text
User

LLM

Application

LLM

Answer
```

Two model calls.

With three tools:

```text
User

LLM

Application

LLM

Application

LLM

Application

LLM

Answer
```

Every additional tool increases latency.

---

### Why Does Cost Increase?

Every API call sends the complete conversation.

Initially:

```text
User
```

Next iteration:

```text
User

Assistant

Tool Result
```

Later:

```text
User

Assistant

Tool Result

Assistant

Tool Result
```

The conversation becomes larger after every iteration.

Therefore:

- More API calls.
- More input tokens.
- Higher cost.

This is why long-running agents become expensive.

---

### Why Is Message Growth Important?

The application always resends the complete conversation.

Example:

Iteration 1

```text
User
```

Iteration 2

```text
User

Assistant

Tool Result
```

Iteration 3

```text
User

Assistant

Tool Result

Assistant

Tool Result
```

Every iteration sends a larger request than the previous one.

This increases:

- Token usage.
- Cost.
- Latency.

Eventually the conversation may approach the model's context window.

---

### How Do Production Systems Handle Long Conversations?

When conversations become very large, production systems usually:

- Remove old messages.
- Summarize earlier conversations.
- Keep only important information.

Otherwise the conversation will eventually exceed the model's context window.

---

### How Do We Debug an Agent?

When an agent fails, inspect the conversation first.

Typical failures include:

#### Tool Never Called

Possible causes:

- Poor system prompt.
- Poor tool description.

---

#### Wrong Tool Called

Possible causes:

- Ambiguous tool descriptions.
- Similar tool names.

---

#### Invalid Arguments

Possible causes:

- Weak schema.
- Missing validation.

---

#### Tool Failed

Possible causes:

- Python exception.
- Database failure.
- API timeout.

This is not an LLM problem.

---

#### Infinite Loop

Possible causes:

- Poor prompting.
- Missing stopping condition.
- Incorrect tool results.

The solution is usually better prompts, improved tool design, or a lower `max_steps`.

---

### Why Is Logging Important?

Every iteration should be logged.

Example:

```text
Step

2

------------------------

Tool

search_knowledge_base

------------------------

Latency

48 ms

------------------------

Status

Success
```

Logs help engineers:

- Debug failures.
- Measure performance.
- Audit tool usage.
- Analyze costs.

---

### Why Is This Better Than One Huge Prompt?

Without an agent:

The LLM must answer using only the initial prompt.

With an agent:

The LLM can:

- Gather information.
- Validate customer data.
- Search documents.
- Call multiple tools.
- Reason using fresh information.
- Produce a much more accurate answer.

This is why modern AI assistants use agent loops instead of relying on a single model call.

---

### Production Engineer's Perspective

Think of an agent as a controlled execution loop.

Every iteration follows exactly the same pattern:

1. Send the complete conversation.
2. Let the LLM reason.
3. Check `stop_reason`.
4. Execute any requested tools.
5. Append the results.
6. Repeat.

The application controls:

- The conversation history.
- Tool execution.
- Security.
- Validation.
- Logging.
- Loop termination.

The LLM controls only one thing:

> **Deciding what should happen next.**

This clear separation of responsibilities is what makes production AI agents reliable, secure, and scalable.

---

### Lead-Level Interview Questions

**Basic**

- Q: Does the model remember previous turns in a tool-calling conversation on its own?  
  A: No — the model has no memory between calls. The entire conversation history, including every prior tool request and its result, must be explicitly included in the `messages` list sent with every single API call. If something isn't in that list, the model has no way to know it happened.

- Q: What does a `tool_use_id` do, and why does it matter?  
  A: It links a specific `tool_result` back to the specific `tool_use` request it's answering. This matters especially once a single model response can request more than one tool call at once — without this linkage, there would be no reliable way to tell the model which result corresponds to which of its multiple requests.

**Intermediate**

- Q: Walk through what would go wrong if a developer forgot to append the assistant's own response to the messages list before appending the tool's result.  
  A: The next API call would include a `tool_result` message with no corresponding prior `tool_use` request visible anywhere in the conversation history sent to the model — since the model only knows what's literally in the messages list, this creates an inconsistent, malformed conversation from the model's perspective, which the API may reject or which could produce confused, incoherent behavior if accepted.

- Q: Why does an ever-growing `messages` list have a compounding, not just additive, effect on cost?  
  A: Because the entire history is resent on every single call, not just the newest message — a conversation that's gone through several tool-call round trips resends all of that accumulated history again on every subsequent call. This means the cost of each individual call increases as the conversation continues, not just the total number of calls made.

**Advanced**

- Q: How does `stop_reason` factor into robust tool-calling loop design, and what should happen if it's not `"tool_use"` when a tool call was expected?  
  A: `stop_reason` tells the calling code why the model stopped generating for this turn — checking it explicitly (rather than assuming a tool call always happened) is a defensive practice, since a model can, for various reasons, produce plain text instead of the expected structured tool call. A well-designed loop should handle this case explicitly — this project's existing code returns an explicit `"unexpected_text_response"` status rather than attempting to parse a tool call that isn't actually present, avoiding a crash or an incorrect assumption about what the model did.

- Q: Design a strategy for managing message-list growth in a long-running, many-tool-call agent conversation, balancing the mechanical requirement that the model needs its history against the cost of resending an ever-larger list.  
  A: This directly extends Chapter 8 Topic 6's multi-turn conversation-management discussion, now understood as a consequence of the exact mechanics covered in this topic — options include truncating older messages (accepting the risk of losing early context, as Chapter 8 Topic 6 warned), summarizing older tool-call-and-result exchanges into a more compact representation before resending, or setting a hard `max_steps`-style limit as this project currently does. The right choice depends on how often conversations actually approach the point where message-list size becomes a measured problem, rather than being decided preemptively without that evidence.

**Scenario-based**

- Q: A production agent conversation involving several tool calls produces a confusing final answer that seems to ignore an earlier tool result. Where would you look first?  
  A: Given that the model has no memory beyond what's literally in the `messages` list, the first and most direct diagnostic step is to print or log the exact `messages` list as it was sent on the final call — inspect whether the earlier tool's result actually made it into that list, whether it's correctly linked via `tool_use_id`, and whether the assistant's own prior response was correctly appended before that result. Since the model's behavior is fully determined by what it can actually see in that list, this mechanical inspection is more reliable than speculating about the model's "reasoning" from the outside.

**Follow-up questions to expect:**

- "How would you handle a scenario where a single model response requests multiple tool calls at once?"
- "What would you log to make debugging a malformed messages list fast and straightforward?"


### Module 1: The Growing Messages List, Made Concrete

Build a real, working multi-turn tool loop (simulated model, real message mechanics) and print the ACTUAL messages list at every step to show it really does grow and get resent in full.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: The growing messages list -- made concrete, not abstract
#
# We cannot call the real Claude API offline. The MODEL RESPONSES below
# are simulated. Everything else -- the messages list construction,
# growth, and resending -- is REAL, exactly matching this project's
# actual run_agent() mechanics.
# ------------------------------------------------------------------

import json

def simulate_model_call(messages: list, step: int) -> dict:
    """Simulates client.messages.create(). Returns a response object
    shaped exactly like the real API: stop_reason + content blocks."""
    if step == 0:
        return {
            "stop_reason": "tool_use",
            "content": [{"type": "tool_use", "id": "toolu_001",
                          "name": "validate_fd_reference",
                          "input": {"reference_number": "BJ2019FD7717"}}],
        }
    else:
        return {
            "stop_reason": "tool_use",
            "content": [{"type": "tool_use", "id": "toolu_002",
                          "name": "finalize_classification",
                          "input": {"label": "FD", "reason": "Customer asking about FD status."}}],
        }


def validate_fd_reference(reference_number: str) -> dict:
    return {"found": True, "customer_name": "Shobha Chopra", "status": "Closed (Premature)"}


messages = [{"role": "user", "content": "Subject: Help\n\nBody: My FD BJ2019FD7717 status please."}]

print("=" * 70)
print("MODULE 1: MESSAGES LIST GROWTH, STEP BY STEP")
print("=" * 70)

for step in range(2):
    print(f"\n--- STEP {step}: messages list has {len(messages)} item(s) BEFORE this call ---")
    for i, m in enumerate(messages):
        content_preview = str(m["content"])[:70]
        msg_role = m["role"]
        print(f"  [{i}] role={msg_role}: {content_preview}...")

    # this is the ACTUAL, REAL call -- sending the FULL messages list
    response = simulate_model_call(messages, step)
    stop_reason = response["stop_reason"]
    print(f"  Model's stop_reason: {stop_reason}")

    # REAL mechanic: append the assistant's own response FIRST
    messages.append({"role": "assistant", "content": response["content"]})

    tool_result_blocks = []
    for block in response["content"]:
        if block["type"] != "tool_use":
            continue
        if block["name"] == "validate_fd_reference":
            result = validate_fd_reference(block["input"]["reference_number"])
            tool_result_blocks.append({"type": "tool_result", "tool_use_id": block["id"],
                                        "content": json.dumps(result)})
        elif block["name"] == "finalize_classification":
            finalize_input = block["input"]
            print(f"\n  Model finalized: {finalize_input}")

    if tool_result_blocks:
        messages.append({"role": "user", "content": tool_result_blocks})

print(f"\nFinal messages list length: {len(messages)} items")
print("Notice: EVERY step re-sent the ENTIRE messages list up to that point --")
print("this is the exact, real mechanic, not a simplification.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: MESSAGES LIST GROWTH, STEP BY STEP

--- STEP 0: messages list has 1 item(s) BEFORE this call ---
  [0] role=user: Subject: Help

Body: My FD BJ2019FD7717 status please....
  Model's stop_reason: tool_use

--- STEP 1: messages list has 3 item(s) BEFORE this call ---
  [0] role=user: Subject: Help

Body: My FD BJ2019FD7717 status please....
  [1] role=assistant: [{'type': 'tool_use', 'id': 'toolu_001', 'name': 'validate_fd_referenc...
  [2] role=user: [{'type': 'tool_result', 'tool_use_id': 'toolu_001', 'content': '{"fou...
  Model's stop_reason: tool_use

  Model finalized: {'label': 'FD', 'reason': 'Customer asking about FD status.'}

Final messages list length: 4 items
Notice: EVERY step re-sent the ENTIRE messages list up to that point --
this is the exact, real mechanic, not a simplification.

Module 1 complete. Run Module 2 next.


### Module 2: The Forgotten-Append Bug, Reproduced

Deliberately skip appending the assistant's response before appending the tool result, and show concretely what breaks in the resulting messages list.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The forgotten-append bug -- reproduced concretely
# ------------------------------------------------------------------

def run_loop_with_bug(initial_message: str) -> list:
    """Deliberately BUGGY version: forgets to append the assistant's
    own response before appending the tool result. Returns the final
    (broken) messages list so we can inspect exactly what's wrong."""
    messages = [{"role": "user", "content": initial_message}]
    response = simulate_model_call(messages, step=0)

    # BUG: we skip this line entirely --
    # messages.append({"role": "assistant", "content": response["content"]})

    tool_result_blocks = []
    for block in response["content"]:
        if block["type"] == "tool_use" and block["name"] == "validate_fd_reference":
            result = validate_fd_reference(block["input"]["reference_number"])
            tool_result_blocks.append({"type": "tool_result", "tool_use_id": block["id"],
                                        "content": json.dumps(result)})

    messages.append({"role": "user", "content": tool_result_blocks})  # appended anyway -- creates the bug
    return messages


def run_loop_correctly(initial_message: str) -> list:
    """The CORRECT version -- appends the assistant response first."""
    messages = [{"role": "user", "content": initial_message}]
    response = simulate_model_call(messages, step=0)

    messages.append({"role": "assistant", "content": response["content"]})  # the required step

    tool_result_blocks = []
    for block in response["content"]:
        if block["type"] == "tool_use" and block["name"] == "validate_fd_reference":
            result = validate_fd_reference(block["input"]["reference_number"])
            tool_result_blocks.append({"type": "tool_result", "tool_use_id": block["id"],
                                        "content": json.dumps(result)})

    messages.append({"role": "user", "content": tool_result_blocks})
    return messages


buggy_messages = run_loop_with_bug("Subject: Help\n\nBody: FD BJ2019FD7717 status?")
correct_messages = run_loop_correctly("Subject: Help\n\nBody: FD BJ2019FD7717 status?")

print("=" * 70)
print("MODULE 2: FORGOTTEN-APPEND BUG -- CONCRETE COMPARISON")
print("=" * 70)

print(f"\nBUGGY messages list ({len(buggy_messages)} items):")
for i, m in enumerate(buggy_messages):
    role_val = m["role"]
    print(f"  [{i}] role={role_val}")

print(f"\nCORRECT messages list ({len(correct_messages)} items):")
for i, m in enumerate(correct_messages):
    role_val = m["role"]
    print(f"  [{i}] role={role_val}")

# REAL check: does the buggy version have a tool_use block anywhere
# for the tool_result to be answering?
def has_matching_tool_use(messages: list, tool_use_id: str) -> bool:
    for m in messages:
        if m["role"] == "assistant":
            for block in m["content"]:
                if isinstance(block, dict) and block.get("type") == "tool_use" and block.get("id") == tool_use_id:
                    return True
    return False

buggy_tool_result_id = buggy_messages[-1]["content"][0]["tool_use_id"]
correct_tool_result_id = correct_messages[-1]["content"][0]["tool_use_id"]

buggy_has_match = has_matching_tool_use(buggy_messages, buggy_tool_result_id)
correct_has_match = has_matching_tool_use(correct_messages, correct_tool_result_id)

print(f"\nBUGGY: does messages list contain the assistant's original tool_use")
print(f"  request that this tool_result is supposedly answering? {buggy_has_match}")
print(f"CORRECT: does messages list contain that original tool_use request? {correct_has_match}")

if not buggy_has_match and correct_has_match:
    print("\nCONFIRMED: the buggy version sends a tool_result with NO visible")
    print("record of the original request it's answering -- exactly the")
    print("malformed conversation state the theory describes, reproduced")
    print("here as a real, checkable structural difference in the messages list.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: FORGOTTEN-APPEND BUG -- CONCRETE COMPARISON

BUGGY messages list (2 items):
  [0] role=user
  [1] role=user

CORRECT messages list (3 items):
  [0] role=user
  [1] role=assistant
  [2] role=user

BUGGY: does messages list contain the assistant's original tool_use
  request that this tool_result is supposedly answering? False
CORRECT: does messages list contain that original tool_use request? True

CONFIRMED: the buggy version sends a tool_result with NO visible
record of the original request it's answering -- exactly the
malformed conversation state the theory describes, reproduced
here as a real, checkable structural difference in the messages list.

Module 2 complete. Run Module 3 next.


### Module 3: Compounding Cost of Message-List Growth

Measure the REAL token-count growth across multiple tool-call round trips, demonstrating the compounding (not just additive) cost effect.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Compounding cost of message-list growth, measured
# ------------------------------------------------------------------

def estimate_tokens(obj) -> int:
    """Rough character-based token estimate (same approach as
    Chapter 8 Topic 1), applied to the full serialized messages list."""
    return len(json.dumps(obj)) // 4


def run_multi_step_loop(n_steps: int) -> list:
    """Runs a simulated multi-step tool-calling loop for n_steps,
    recording the ACTUAL token cost of the messages list sent on
    EACH step -- this is what makes the compounding effect measurable
    with real numbers, not just asserted."""
    messages = [{"role": "user", "content": "Subject: Help\n\nBody: Multiple questions about my FD account."}]
    token_costs_per_step = []

    for step in range(n_steps):
        token_costs_per_step.append(estimate_tokens(messages))  # cost of THIS call

        response = simulate_model_call(messages, step=0)  # reuse simple simulated response
        messages.append({"role": "assistant", "content": response["content"]})

        tool_result_blocks = []
        for block in response["content"]:
            if block["type"] == "tool_use" and block["name"] == "validate_fd_reference":
                result = validate_fd_reference(block["input"]["reference_number"])
                tool_result_blocks.append({"type": "tool_result", "tool_use_id": block["id"],
                                            "content": json.dumps(result)})
        if tool_result_blocks:
            messages.append({"role": "user", "content": tool_result_blocks})

    return token_costs_per_step


token_costs = run_multi_step_loop(n_steps=5)

print("=" * 70)
print("COMPOUNDING COST OF MESSAGE-LIST GROWTH (real token estimates)")
print("=" * 70)

print(f"{'Step':<6} | {'Tokens sent THIS call':>22} | {'Cumulative tokens sent':>23}")
print("-" * 60)
cumulative = 0
for step, tokens in enumerate(token_costs):
    cumulative += tokens
    print(f"{step:<6} | {tokens:>22} | {cumulative:>23}")

print(f"\nStep 0 sent {token_costs[0]} tokens. Step {len(token_costs)-1} sent {token_costs[-1]} tokens --")
growth_factor = token_costs[-1] / token_costs[0] if token_costs[0] > 0 else float("inf")
print(f"a {growth_factor:.1f}x increase, purely from accumulated history, NOT from")
print(f"any single message getting longer. This is the concrete, measured")
print(f"version of the theory's compounding-cost claim: TOTAL tokens sent")
print(f"across all {len(token_costs)} steps is {cumulative}, not just {len(token_costs)} times the first")
print(f"step's cost -- each step resends everything before it, every time.")

print("\nModule 3 complete. All Chapter 10 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  The model has NO memory of its own -- the ENTIRE messages list is
  resent, in full, on EVERY single API call. This is THE core mechanic
  underlying everything else in tool-calling.

  Forgetting to append the assistant's own response before appending
  the tool result creates a malformed conversation -- demonstrated
  concretely: no matching tool_use exists for the tool_result to answer.

  tool_use_id links a specific tool_result back to the specific
  tool_use request it answers -- critical once multiple tools are
  requested in a single turn.

  Message-list growth has a COMPOUNDING cost effect, not just additive
  -- demonstrated with real token counts: total cost across N steps is
  much more than N times a single step's cost.

  stop_reason must be checked explicitly -- never assume a tool call
  happened without verifying it.
""")


COMPOUNDING COST OF MESSAGE-LIST GROWTH (real token estimates)
Step   |  Tokens sent THIS call |  Cumulative tokens sent
------------------------------------------------------------
0      |                     23 |                      23
1      |                    110 |                     133
2      |                    197 |                     330
3      |                    284 |                     614
4      |                    370 |                     984

Step 0 sent 23 tokens. Step 4 sent 370 tokens --
a 16.1x increase, purely from accumulated history, NOT from
any single message getting longer. This is the concrete, measured
version of the theory's compounding-cost claim: TOTAL tokens sent
across all 5 steps is 984, not just 5 times the first
step's cost -- each step resends everything before it, every time.

Module 3 complete. All Chapter 10 Topic 2 modules done.

CHAPTER 10 TOPIC 2 -- KEY POINTS TO REMEMBER

  The model has NO memory of its own -- the ENTIRE messages l